# Datamine BHCOUNT Process Example

This notebook demonstrates how to configure and run the **`bhcount`** process wrapper in `dmstudio`.

## Process Description

## BHCOUNT

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This command counts the number of boreholes used to estimate each model cell. This information can then be used to assist in assigning a reserve category to the model cells.

The two input files, MODELIN and SAMPIN, must have been created by the grade estimation process [ESTIMA](<estima.md>):

* **MODELIN** \- the output model from ESTIMA.

* **SAMPIN** \- the ouput sample file from ESTIMA.

When running [ESTIMA](<estima.md>) to create these two files, the following options must have been specified:

1. **Key Field**
This restricts the number of samples per key field, where the key field will usually be BHID. Although this option must have been selected when running ESTIMA, it need not actually affect the results. If you do not want to restrict the number of samples per key field, then select the maximum number to be greater than or equal to the maximum number of samples in the search volume. This will ensure that it has no effect. The reason for using the option is so that the key field (BHID) is copied to the sample output file.

2. **Sample Output File**
A sample output file must be created.

The input to **BHCOUNT** includes the following:

1. Key Field (**KEY**)
The same key field must be specified as for the corresponding ESTIMA run.

2. Grade Value (**VALUE**)
If multiple values of the VALUE_OU field were specified in the Estimation Parameter File for the ESTIMA run then there will be multiple values in the FIELD field of the SAMPIN file. However BHCOUNT can only process one value, which can be selected using the VALUE field. Note that this usage is different from normal field prompts as the input is a field value, not a field name. Therefore it cannot be selected from a dropdown list but must be entered manually. If the FIELD field in the SAMPIN file only contains one value then it does not have to be entered as it will be selected automatically. Note also that the field value is restricted to 24 characters.

3. Parent Parameter (**PARENT**)
The same value of parameter PARENT that was used for the ESTIMA run must also be specified for the BHCOUNT run. Failure to do this will lead to incorrect values in the MODELOUT file.

The output model from BHCOUNT will be the same as the input model, but with the extra field N-BHID.

### Input Files:

* **modelin** (Block Model):
  Input model file. This file is the output MODEL file from ESTIMA
  Required=Yes

* **sampin** (Table):
  Input sample file. This file is the SAMPOUT file from ESTIMA. The run of ESTIMA which
  created the file must have used the following options: - the key field option was
  selected. - only one output grade field was created.
  Required=Yes

### Output Files:

* **modelout** (Block Model):
  Output model file. This is the same as the input model, but with the extra field

## N-BHID.

  Required=Yes

### Fields:

* **key** (Any : SAMPIN):
  Name of the field containing the drillhole identification code. This is the field
  specified as the KEY field when running ESTIMA. This will usually be BHID, which is the
  default.
  Default=BHID
  Required=Yes

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('bhcount')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute bhcount
print("Running bhcount...")
dm_cmd.bhcount(
    modelin_i='t_mod1',  # required
    sampin_i='t_input_file',  # required
    modelout_o='t_bhcount_out',  # required
    key_f=['BHID'],  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("bhcount execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_bhcount_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")